In [ ]:
# pip install requests

#Stock events

import os
import requests
import pandas as pd

your_key = os.getenv("X_API_KEY")
print(your_key)
url = "https://gateway.datacore.vn/data/ds/search"

headers = {
    "x-api-key": "",
    "Content-Type": "application/json"
}

payload = {
    "dataSetCode": "dataset_stock_events", #name of datasetCode
    "conditions": [],
    "selectFields": [],
    "page": 1, 
    "limit": 54000
}
res = requests.post(url, headers=headers, json=payload)
# print(res)

data = res.json()
# print(data)

df = pd.DataFrame(
    data["data"]["dataDetail"],
    columns=data["data"]["fields"]
)
df

None


,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,21/12/2023,22/12/2023,15/01/2024,UPCOM,2023
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,01/08/2023,02/08/2023,23/08/2023,UPCOM,2023
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,03/04/2023,04/04/2023,30/06/2023,UPCOM,2023
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,15/12/2022,16/12/2022,09/01/2023,UPCOM,2022
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,01/08/2022,02/08/2022,23/08/2022,UPCOM,2022
...,...,...,...,...,...,...,...,...,...,...
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaN,NaN,18/08/2017,UPCOM,2017
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaN,NaN,03/08/2017,UPCOM,2017
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,30/06/2017,03/07/2017,14/08/2017,UPCOM,2017
53660,YTC,2017-06-20,Share Issue,YTC-Issues bonus shares at the ex-rate of 10:1,ISS,30/06/2017,03/07/2017,30/06/2017,UPCOM,2017


In [8]:
df = df.copy()
text_col = ["symbol", "event_name", "event_title", "event_code","exchange"]
for col in text_col:
    df[col] = df[col].astype("string")
date_cols = ["date_exr", "date_record", "date_issue"]
for date in date_cols:
    df[date] = pd.to_datetime(df[date], errors = "coerce", dayfirst = True)
df["date"] = pd.to_datetime(df["date"], errors = "coerce")
df["year"] = pd.to_numeric(df["year"], errors = "coerce").astype("Int64")
df.dtypes

symbol                 string
date           datetime64[us]
event_name             string
event_title            string
event_code             string
date_exr       datetime64[us]
date_record    datetime64[us]
date_issue     datetime64[us]
exchange               string
year                    Int64
dtype: object

In [9]:
#kiểm tra các cột date 
date_year =  (df["date_issue"].dt.year.astype("Int64").value_counts().reset_index(name='count'))
print(date_year)

    date_issue  count
0         2016   9748
1         2017   9662
2         2015   4837
3         2014   3819
4         2018   3063
5         2011   2790
6         2022   2530
7         2012   2303
8         2021   2275
9         2023   2261
10        2019   2260
11        2020   2135
12        1753   2124
13        2010   1941
14        2013   1515
15        2009    260
16        2024    113
17        2025     15
18        2008      4
19        2026      2
20        2004      1
21        2030      1
22        2007      1
23        1947      1
24        2027      1


In [10]:
# clean date sai lệch
df_clean = df.drop(df[(df["date_issue"].dt.year >= 2025) | (df["date_issue"].dt.year <= 2007)].index)
date_year_clean =  (df_clean["date_issue"].dt.year.astype("Int64").value_counts().reset_index(name='count'))
print(date_year_clean)


    date_issue  count
0         2016   9748
1         2017   9662
2         2015   4837
3         2014   3819
4         2018   3063
5         2011   2790
6         2022   2530
7         2012   2303
8         2021   2275
9         2023   2261
10        2019   2260
11        2020   2135
12        2010   1941
13        2013   1515
14        2009    260
15        2024    113
16        2008      4


In [11]:
# Drop duplicate row
df_clean = df_clean.drop_duplicates(subset=["symbol", "date","event_code","event_name"], keep = "first")
df_clean

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022
...,...,...,...,...,...,...,...,...,...,...
53656,YTC,2018-02-01,Cash Dividend,"YTC-Pays Interim 2, 2017 cash dividend at VND1...",DIV,2018-02-02,2018-02-05,2018-02-12,UPCOM,2018
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaT,NaT,2017-08-18,UPCOM,2017
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaT,NaT,2017-08-03,UPCOM,2017
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,2017-06-30,2017-07-03,2017-08-14,UPCOM,2017


In [12]:
#check date.year == year
df_clean[df_clean["date"].dt.year != df_clean["year"]]

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year


In [13]:
event_group_map = {
    'KQQY': 'Financial Reporting',
    'KQCT': 'Financial Reporting',
    'KQSB': 'Financial Reporting',

    'DIV': 'Corporate Action',
    'ISS': 'Corporate Action',
    'AIS': 'Corporate Action',
    'TS': 'Corporate Action',

    'AGME': 'Shareholder Meeting',
    'EGME': 'Shareholder Meeting',
    'AGMR': 'Shareholder Meeting',
    'BALLOT': 'Shareholder Meeting',

    'BCHA': 'Governance',
    'BOME': 'Governance',

    'DDALL': 'Insider Trading',
    'DDIND': 'Insider Trading',
    'DDINS': 'Insider Trading',
    'DDRP': 'Insider Trading',

    'NLIS': 'Listing Status',
    'RETU': 'Listing Status',
    'SUSP': 'Listing Status',
    'MOVE': 'Listing Status',

    'AMEN': 'Corporate Restructuring',
    'MA': 'Corporate Restructuring',

    'OTHE': 'Other'
}

df['event_group'] = df['event_code'].map(event_group_map).fillna('Other')
df.head()

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year,event_group
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023,Corporate Action
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023,Corporate Action
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023,Shareholder Meeting
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022,Corporate Action
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022,Corporate Action


In [14]:
#check logic date_record, date_exr
check_date = (df_clean["date_exr"].notna() & 
              df_clean["date_record"].notna() &
              ~ (df_clean["date_exr"] <= df_clean["date_record"])
             )
df_clean = df_clean.drop(df_clean[check_date].index)
df_clean 

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022
...,...,...,...,...,...,...,...,...,...,...
53656,YTC,2018-02-01,Cash Dividend,"YTC-Pays Interim 2, 2017 cash dividend at VND1...",DIV,2018-02-02,2018-02-05,2018-02-12,UPCOM,2018
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaT,NaT,2017-08-18,UPCOM,2017
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaT,NaT,2017-08-03,UPCOM,2017
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,2017-06-30,2017-07-03,2017-08-14,UPCOM,2017


In [15]:
# check validate date_record, date_exr
mask = (
    df_clean["date_record"].notna() &
    df_clean["date_exr"].notna() &
    ~ ((df_clean["date_record"] - df_clean["date_exr"]).dt.days < 20)
)

df_clean = df_clean.drop(df_clean[mask].index)
df_clean

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022
...,...,...,...,...,...,...,...,...,...,...
53656,YTC,2018-02-01,Cash Dividend,"YTC-Pays Interim 2, 2017 cash dividend at VND1...",DIV,2018-02-02,2018-02-05,2018-02-12,UPCOM,2018
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaT,NaT,2017-08-18,UPCOM,2017
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaT,NaT,2017-08-03,UPCOM,2017
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,2017-06-30,2017-07-03,2017-08-14,UPCOM,2017


In [16]:
#check NA date_record, date_exr
event_date_status = (
    df_clean.assign(
        has_both_dates = df_clean["date_exr"].notna() & df_clean["date_record"].notna()
    )
    .groupby("event_code")
    .agg(
        total_rows=("event_code", "size"),
        full_date_rows=("has_both_dates", "sum")
    )
    .reset_index()
)

event_date_status["missing_date_rows"] = (
    event_date_status["total_rows"] - event_date_status["full_date_rows"]
)

print(event_date_status.sort_values("missing_date_rows", ascending=False))

   event_code  total_rows  full_date_rows  missing_date_rows
6        BOME       10321               0              10321
15       KQQY        9044               0               9044
5        BCHA        5387               0               5387
2         AIS        2316               0               2316
1        AGMR        2186               0               2186
14       KQCT        2016               0               2016
23         TS        1364               0               1364
19       NLIS         746               0                746
9       DDINS         435               0                435
7       DDALL         411               0                411
20       OTHE         390               2                388
16       KQSB         296               0                296
12       EGME         788             557                231
22       SUSP         223               0                223
18       MOVE         218               0                218
3        AMEN         16

In [17]:
#DELETE date_exr, date_record not available
df_clean = df_clean.loc[~((df_clean["event_code"] == "AGME") & df_clean["date_exr"].isna())]
df_clean = df_clean.loc[~((df_clean["event_code"] == "DIV") & df_clean["date_exr"].isna())]
df_clean

,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022
...,...,...,...,...,...,...,...,...,...,...
53656,YTC,2018-02-01,Cash Dividend,"YTC-Pays Interim 2, 2017 cash dividend at VND1...",DIV,2018-02-02,2018-02-05,2018-02-12,UPCOM,2018
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaT,NaT,2017-08-18,UPCOM,2017
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaT,NaT,2017-08-03,UPCOM,2017
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,2017-06-30,2017-07-03,2017-08-14,UPCOM,2017


In [18]:
#
df_clean["event_code_norm"] = df_clean["event_code"]
mask = (
    (df_clean["event_code"] == "OTHE") &
    (df_clean["event_title"].str.contains("dividend", case=False, na=False))
)

df_clean.loc[mask, "event_code_norm"] = "DIV"


In [23]:
# kiểm tra cột event_title 
missing_title = (
    df_clean.groupby("event_code")["event_title"]
    .apply(lambda x: x.isna().sum())
    .reset_index(name = "missing_title_count")
    .sort_values("missing_title_count", ascending = False)
)
print(missing_title)

   event_code  missing_title_count
6        BOME                 5029
15       KQQY                 4528
5        BCHA                 3010
14       KQCT                 1125
1        AGMR                  825
2         AIS                  765
23         TS                  708
19       NLIS                  310
12       EGME                  231
16       KQSB                   69
20       OTHE                   60
22       SUSP                   48
18       MOVE                   44
3        AMEN                   40
21       RETU                    9
0        AGME                    6
9       DDINS                    0
10       DDRP                    0
11        DIV                    0
8       DDIND                    0
13        ISS                    0
7       DDALL                    0
4      BALLOT                    0
17         MA                    0


In [ ]:
# # fill NA text
# if df_clean["event_title"].isna().any():
#     df_clean.loc[df_clean["event_code"] == "MA", "event_title"] = (df_clean["symbol"] + " - Merger and Acquisition")
#     df_clean.loc[df_clean["event_code"] == "AGME", "event_title"] = (df_clean["symbol"] + " - Holds " + df_clean["year"] + " AGM")
#     df_clean.loc[df_clean["event_code"] == "RETU", "event_title"] = (df_clean["symbol"] + " re-listing at " + df_clean["year"])
#     df_clean.loc[df_clean["event_code"] == "AMEN", "event_title"] = ("Changes company' name")
#     df_clean.loc[df_clean["event_code"] == "AGME", "event_title"] = (df_clean["symbol"] + " - Holds " + df_clean["year"] + " AGM")
#     df_clean.loc[df_clean["event_code"] == "MOVE", "event_title"] = ("switch exchange to " + df_clean["exchange"])


,symbol,date,event_name,event_title,event_code,date_exr,date_record,date_issue,exchange,year,event_code_norm
0,A32,2023-12-18,Cash Dividend,A32 - Pay cash dividend 2023 - Interim 1 at VN...,DIV,2023-12-21,2023-12-22,2024-01-15,UPCOM,2023,DIV
1,A32,2023-07-25,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 2 at VN...,DIV,2023-08-01,2023-08-02,2023-08-23,UPCOM,2023,DIV
2,A32,2023-03-17,Annual General Meeting,A32 - Holds 2023 AGM,AGME,2023-04-03,2023-04-04,2023-06-30,UPCOM,2023,AGME
3,A32,2022-12-09,Cash Dividend,A32 - Pay cash dividend 2022 - Interim 1 at VN...,DIV,2022-12-15,2022-12-16,2023-01-09,UPCOM,2022,DIV
4,A32,2022-07-28,Cash Dividend,A32 - Pay cash dividend 2021 - Interim 2 at VN...,DIV,2022-08-01,2022-08-02,2022-08-23,UPCOM,2022,DIV
...,...,...,...,...,...,...,...,...,...,...,...
53656,YTC,2018-02-01,Cash Dividend,"YTC-Pays Interim 2, 2017 cash dividend at VND1...",DIV,2018-02-02,2018-02-05,2018-02-12,UPCOM,2018,DIV
53657,YTC,2017-08-16,Additional Listing,"YTC-Registers 280,000 additional shares",AIS,NaT,NaT,2017-08-18,UPCOM,2017,AIS
53658,YTC,2017-07-26,New listing,YTC-Officially starts trading on UPCoM,NLIS,NaT,NaT,2017-08-03,UPCOM,2017,NLIS
53659,YTC,2017-06-20,Cash Dividend,"YTC - Pays Interim 2, 2016 and Interim 1, 2017...",DIV,2017-06-30,2017-07-03,2017-08-14,UPCOM,2017,DIV


In [ ]:
import pandas as pd
import re

keyword_map = {
    "positive": [
        # điền từ khóa positive
        r".*cash.*dividend.*shares?.*",
        r".*buy.*treasury.*shares?.*"
    ],
    "negative": [
        # điền từ khóa negative
    ],
    "neutral": [
        # điền từ khóa neutral
    ]
}

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def assign_sentiment_label_v2(title):
    text = normalize_text(title)

    if text == "":
        return pd.NA

    matched_labels = []

    for label, patterns in keyword_map.items():
        for pattern in patterns:
            if re.search(pattern, text):
                matched_labels.append(label)
                break
                
    matched_labels = list(set(matched_labels))

    if len(matched_labels) == 1:
        return matched_labels[0]
    elif len(matched_labels) > 1:
        return "mixed"
    else:
        return "unknown"

df_clean["sentiment_label"] = df_clean["event_title"].apply(assign_sentiment_label_v2)

In [26]:
symbol_event_freq = (
    df_clean.groupby("symbol")["event_code"]
    .nunique()
    .reset_index(name="count")
)
print(symbol_event_freq.loc[symbol_event_freq["count"] > 17, "symbol"])     

1      AAA
226    DNM
277    FIT
578    PAN
604    PHC
637    PPP
Name: symbol, dtype: string


In [ ]:
#Narrative index
import os
import requests
import pandas as pd

your_key = os.getenv("X_API_KEY")
print(your_key)
url = "https://gateway.datacore.vn/data/ds/search"

headers = {
    "x-api-key": "",
    "Content-Type": "application/json"
}

payload = {
    "dataSetCode": "Narrative Index", 
    "conditions": [],
    "selectFields": [],
    "page": 1, 
    "limit": 6300
}
res = requests.post(url, headers=headers, json=payload)
# print(res)

data = res.json()
print(data)

# Example
df = pd.DataFrame(
    data["data"]["dataDetail"],
    columns=data["data"]["fields"]
)
df


None
{'status': True, 'message': 'success', 'httpCode': 200, 'errorCode': 'ok', 'data': {'fields': ['month', 'year', 'ticker', 'topic', 'w'], 'dataDetail': [['2015-01', 2015, 'VNINDEX', 'Education', 3.618157927], ['2015-01', 2015, 'VNINDEX', 'Economy', 2.643936351], ['2015-01', 2015, 'VNINDEX', 'Weather', 0.782797702], ['2015-01', 2015, 'VNINDEX', 'Finance', 0.737691475], ['2015-01', 2015, 'VNINDEX', 'Politics', 0.631826212], ['2015-01', 2015, 'VNINDEX', 'Law', 0.251158634], ['2015-02', 2015, 'VNINDEX', 'Media', 188.8122955], ['2015-02', 2015, 'VNINDEX', 'Epidemic', 159.9336431], ['2015-02', 2015, 'VNINDEX', 'Energy', 78.04814791], ['2015-02', 2015, 'VNINDEX', 'Corruption', 65.55725162], ['2015-02', 2015, 'VNINDEX', 'Banking', 60.82423691], ['2015-02', 2015, 'VNINDEX', 'Science', 53.59119334], ['2015-02', 2015, 'VNINDEX', 'Healthcare', 45.69532659], ['2015-02', 2015, 'VNINDEX', 'Defense', 45.47524483], ['2015-02', 2015, 'VNINDEX', 'Administration', 41.95828788], ['2015-02', 2015, 'VNIN

,month,year,ticker,topic,w
0,2015-01,2015,VNINDEX,Education,3.618158
1,2015-01,2015,VNINDEX,Economy,2.643936
2,2015-01,2015,VNINDEX,Weather,0.782798
3,2015-01,2015,VNINDEX,Finance,0.737691
4,2015-01,2015,VNINDEX,Politics,0.631826
...,...,...,...,...,...
6218,2023-12,2023,VNINDEX,Real estate,5.961272
6219,2023-12,2023,VNINDEX,History,5.282165
6220,2023-12,2023,VNINDEX,Economy,5.049362
6221,2023-12,2023,VNINDEX,Defense,5.010488
